In [1]:
#@title 1. Clustering: Descubrimiento de Familias de Diseño
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import sys
import os

# 1. Configuración de entorno
sys.path.append(os.path.abspath('..'))
from src.utils import set_global_style
set_global_style()

print("🧩 Iniciando Modelado No Supervisado (Clustering)...\n")

# 2. Carga de datos (Usamos el dataset completo, no cruzamos con producción)
ruta_procesados = os.path.join("..", "data", "processed", "pozos_analitico.csv")
df_clustering = pd.read_csv(ruta_procesados)
print(f"📥 Datos cargados: {len(df_clustering)} pozos en el alcance total.")

# 3. Feature Engineering Específico: Ratio de Arena Importada
# Esta es la variable que tu README destaca como clave para definir una familia entera
arena_imp = df_clustering.get('arena_bombeada_importada_tn', 0).fillna(0)
arena_tot = df_clustering['arena_total_tn'].replace(0, np.nan) # Evitar división por cero
df_clustering['ratio_arena_importada'] = (arena_imp / arena_tot).fillna(0)

# Seleccionamos las variables de intensidad operativa
vars_clustering = ['arena_por_metro', 'agua_por_metro', 'espaciamiento_etapas', 'ratio_arena_importada']
df_ml = df_clustering.dropna(subset=vars_clustering).copy()

# 4. Tratamiento de Outliers (El hallazgo técnico de la preentrega 4)
print("✂️ Acotando valores extremos (1% - 99%) para proteger a K-Means...")
for col in vars_clustering:
    q_low = df_ml[col].quantile(0.01)
    q_hi  = df_ml[col].quantile(0.99)
    df_ml[col] = df_ml[col].clip(q_low, q_hi)

# 5. Estandarización de las variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_ml[vars_clustering])

# 6. Entrenamiento de K-Means
# k=4 familias, n_init=200 para máxima estabilidad (según documentación del proyecto)
print("🤖 Ajustando K-Means con k=4 familias (n_init=200)...")
kmeans = KMeans(n_clusters=4, n_init=200, random_state=42)
df_ml['cluster'] = kmeans.fit_predict(X_scaled)
df_ml['cluster'] = df_ml['cluster'].astype(str) 

# Asignamos nombres genéricos temporales (en la defensa los llamarás "Intensivo moderno", etc.)
df_ml['Familia de Diseño'] = 'Familia ' + df_ml['cluster']

# 7. Reducción de Dimensionalidad (PCA) para Visualización en 2D
pca = PCA(n_components=2)
componentes = pca.fit_transform(X_scaled)
df_ml['PCA1'] = componentes[:, 0]
df_ml['PCA2'] = componentes[:, 1]

# 8. Visualización A: Mapa de Familias (PCA)
fig_pca = px.scatter(
    df_ml, x='PCA1', y='PCA2', color='Familia de Diseño',
    hover_data=['idpozo', 'anio_fractura', 'operadora', 'longitud_rama_horizontal'],
    title='Mapa de Familias de Diseño en Vaca Muerta (Proyección PCA 2D)',
    labels={'PCA1': 'Componente Principal 1 (Intensidad General)', 'PCA2': 'Componente Principal 2 (Composición)'},
    opacity=0.7, height=600, color_discrete_sequence=px.colors.qualitative.Prism
)
fig_pca.update_layout(legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center", title=""))
fig_pca.show()

# 9. Visualización B: Evolución Histórica de las Familias
# Esto demuestra que aunque el modelo no vio el año, las familias son cronológicas
df_evol = df_ml.groupby(['anio_fractura', 'Familia de Diseño']).size().reset_index(name='Cantidad de Pozos')

fig_evol = px.bar(
    df_evol, x='anio_fractura', y='Cantidad de Pozos', color='Familia de Diseño',
    title='Evolución Histórica: Cómo cambió el diseño sin decírselo al modelo',
    labels={'anio_fractura': 'Año de Fractura'},
    height=500, color_discrete_sequence=px.colors.qualitative.Prism
)
fig_evol.update_layout(
    barmode='stack', 
    legend=dict(orientation="h", y=-0.2, x=0.5, xanchor="center", title="")
)
fig_evol.show()

🧩 Iniciando Modelado No Supervisado (Clustering)...

📥 Datos cargados: 2603 pozos en el alcance total.
✂️ Acotando valores extremos (1% - 99%) para proteger a K-Means...
🤖 Ajustando K-Means con k=4 familias (n_init=200)...
